# Volume Surge + Breakout on SPY
## Strategy Brief -- 3-5 plain-English sentences on signal, prediction, trade logic, results.
The Volume Surge + Breakout strategy aims to capture momentum in SPY by identifying significant volume increases that coincide with price breakouts. The hypothesis is that a surge in volume can indicate strong interest in a security, often preceding a substantial price move. The strategy enters a long position when a volume surge is detected alongside a price breakout above a recent high. The exit occurs when the price falls below a trailing stop or a reversal in the volume trend. Historically, this strategy can outperform simple buy-and-hold by capturing short-term price movements.
## References
- https://www.omnicalculator.com/math/volume

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## PHASE 1 - Trading Context
In this phase, we define the parameters and constants needed for our trading strategy. These include the lookback periods for volume and price, the threshold for volume surge, and any other necessary configuration.

In [ ]:
VOLUME_LOOKBACK = 20
PRICE_LOOKBACK = 20
VOLUME_SURGE_THRESHOLD = 1.5
TRAILING_STOP_PERCENT = 0.05
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'

## PHASE 2 - Data Exploration
We will download historical SPY data using yfinance, calculate the volume surge indicator, and overlay it on the price chart to visualize potential entry points.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

spy = yf.download('SPY', start=START_DATE, end=END_DATE)
spy['Volume_MA'] = spy['Volume'].rolling(window=VOLUME_LOOKBACK).mean()
spy['Volume_Surge'] = spy['Volume'] / spy['Volume_MA']

plt.figure(figsize=(14, 7))
plt.plot(spy['Close'], label='SPY Close Price')
plt.plot(spy['Volume_Surge'], label='Volume Surge', linestyle='--')
plt.axhline(y=VOLUME_SURGE_THRESHOLD, color='r', linestyle='-')
plt.legend()
plt.title('SPY Price and Volume Surge')
plt.show()

## PHASE 3 - Strategy Engineering
We will define the signal for entering and exiting trades. A long position is initiated when the volume surge exceeds the threshold and the price breaks out above the recent high. The position is exited when the price falls below a trailing stop.

In [ ]:
spy['Signal'] = 0
spy['High_Price_Lookback'] = spy['Close'].rolling(window=PRICE_LOOKBACK).max()

for i in range(len(spy)):
    if spy['Volume_Surge'].iloc[i] > VOLUME_SURGE_THRESHOLD and spy['Close'].iloc[i] > spy['High_Price_Lookback'].iloc[i-1]:
        spy['Signal'].iloc[i] = 1
    elif spy['Close'].iloc[i] < spy['Close'].iloc[i-1] * (1 - TRAILING_STOP_PERCENT):
        spy['Signal'].iloc[i] = 0

spy['Position'] = spy['Signal'].shift(1).fillna(0)

## PHASE 4 - Coding & Backtesting
We will now backtest the strategy by calculating daily returns and plotting the equity curve to visualize the strategy's performance over time.

In [ ]:
spy['Daily_Return'] = spy['Close'].pct_change()
spy['Strategy_Return'] = spy['Daily_Return'] * spy['Position']
spy['Equity_Curve'] = (1 + spy['Strategy_Return']).cumprod()

plt.figure(figsize=(14, 7))
plt.plot(spy['Equity_Curve'], label='Strategy Equity Curve')
plt.plot((1 + spy['Daily_Return']).cumprod(), label='Buy and Hold Equity Curve')
plt.legend()
plt.title('Equity Curve Comparison')
plt.show()

## PHASE 5 - Performance Evaluation
In this phase, we calculate key performance metrics such as CAGR, Sharpe Ratio, Sortino Ratio, Calmar Ratio, and maximum drawdown to evaluate the strategy's performance compared to a buy-and-hold approach.

In [ ]:
def calculate_performance_metrics(df):
    cagr = (df['Equity_Curve'].iloc[-1])**(1/(len(df)/252)) - 1
    sharpe_ratio = df['Strategy_Return'].mean() / df['Strategy_Return'].std() * np.sqrt(252)
    downside_std = df[df['Strategy_Return'] < 0]['Strategy_Return'].std()
    sortino_ratio = df['Strategy_Return'].mean() / downside_std * np.sqrt(252)
    max_drawdown = (df['Equity_Curve'].cummax() - df['Equity_Curve']).max()
    calmar_ratio = cagr / max_drawdown
    
    return cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

strategy_metrics = calculate_performance_metrics(spy)
buy_and_hold_metrics = calculate_performance_metrics(spy.assign(Strategy_Return=spy['Daily_Return']))

comparison_table = pd.DataFrame({
    'Metric': ['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Max Drawdown'],
    'Strategy': strategy_metrics,
    'Buy and Hold': buy_and_hold_metrics
})

print(comparison_table)

## PHASE 6 - Deploy & Monitor
Finally, we will create a function to download the last 60 days of SPY data, compute today's signal, and print the recommended position.

In [ ]:
def get_latest_signal():
    recent_spy = yf.download('SPY', period='60d')
    recent_spy['Volume_MA'] = recent_spy['Volume'].rolling(window=VOLUME_LOOKBACK).mean()
    recent_spy['Volume_Surge'] = recent_spy['Volume'] / recent_spy['Volume_MA']
    recent_spy['High_Price_Lookback'] = recent_spy['Close'].rolling(window=PRICE_LOOKBACK).max()
    
    if (recent_spy['Volume_Surge'].iloc[-1] > VOLUME_SURGE_THRESHOLD and
        recent_spy['Close'].iloc[-1] > recent_spy['High_Price_Lookback'].iloc[-2]):
        return "Long"
    elif recent_spy['Close'].iloc[-1] < recent_spy['Close'].iloc[-2] * (1 - TRAILING_STOP_PERCENT):
        return "Exit"
    else:
        return "Hold"

print("Today's Position Recommendation: ", get_latest_signal())